# RAG System - Query & Retrieval Demo

This notebook demonstrates the query and retrieval stage of the RAG pipeline:
1. **Load Vector Store** - Load the pre-built FAISS index
2. **Test Retrieval** - Query the vector store and inspect results
3. **LLM Response** - Generate answers with citations (TODO)
4. **Evaluation** - Metrics on test questions (TODO)

Prerequisites: Run `01_indexing_demo.ipynb` first to create the vector store.

---
## Setup

In [2]:
# Imports
import os
import numpy as np
from pathlib import Path
import json

from rag_system.vector_store import FAISSVectorStore, VectorStoreConfig, SearchResult
from rag_system.embeddings import Embedder, EmbeddingConfig
from rag_system.models import DocumentChunk

In [3]:
# Project paths
PROJECT_ROOT = Path.cwd()
INDEX_DIR = PROJECT_ROOT / "faiss_index"

print(f"Project root:     {PROJECT_ROOT}")
print(f"Index directory:  {INDEX_DIR}")
print(f"Index exists:     {INDEX_DIR.exists()}")

Project root:     c:\Development\git\rag_achiles
Index directory:  c:\Development\git\rag_achiles\faiss_index
Index exists:     True


---
## 1. Load Vector Store

Load the pre-built FAISS index and metadata from disk.

In [4]:
# Load vector store (auto-loads existing index if found)
print("Loading vector store...\n")

threshold = 0.6 # set similiratity threshold
vs_config = VectorStoreConfig(similarity_threshold=threshold)
vector_store = FAISSVectorStore(config=vs_config)

print(f"✓ Vector store loaded successfully!")
print(f"\nStatistics:")
stats = vector_store.get_stats()
for key, value in stats.items():
    print(f"  {key:20s}: {value}")

Loading vector store...

Loaded index with 619 vectors from faiss_index\vector_index.faiss
✓ Vector store loaded successfully!

Statistics:
  total_vectors       : 619
  dimension           : 384
  index_type          : FAISS-IndexFlatIP
  total_chunks        : 619
  similarity_threshold: 0.6
  index_path          : faiss_index\vector_index.faiss
  metadata_path       : faiss_index\metadata.pkl


### Initialize Embedder

Load the same embedding model used during indexing to embed queries.

In [5]:
# Load embedder (same model as indexing)
print("Loading embedder...\n")

embedder = Embedder()

print(f"\nEmbedder ready:")
print(f"  Model:     {embedder.config.model_name}")
print(f"  Dimension: {embedder.dimension}")

Loading embedder...

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2...


c:\Development\environments\aida-venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✓ Model loaded (dimension: 384)

Embedder ready:
  Model:     paraphrase-multilingual-MiniLM-L12-v2
  Dimension: 384


---
## 2. Test Retrieval (No LLM)

Let's test the vector store retrieval with sample queries to verify semantic search works correctly.

### Helper Functions

Create reusable functions for querying and analyzing results.

In [6]:
def query_and_display(query: str, top_k: int = 5, show_full_text: bool = False, show_rerank: bool = True):
    """
    Query vector store and display results.
    
    Args:
        query: Search query
        top_k: Number of results to retrieve
        show_full_text: If True, show complete chunk text; if False, show preview
        show_rerank: If True, show rerank score when available
    """
    query_emb = embedder.embed_query(query)
    results = vector_store.search(query_emb, top_k=top_k)
    
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)
    print(f"Found {len(results)} results (top-{top_k})\n")
    
    if not results:
        print("No results found above similarity threshold.\n")
        return results
    
    for i, result in enumerate(results, 1):
        chunk = result.chunk
        score_str = f"Score: {result.score:.4f}"
        
        # Show rerank score if available and requested
        if show_rerank and result.rerank_score is not None:
            score_str += f" | Rerank: {result.rerank_score:.3f}"
        
        print(f"[{i}] {score_str} | {chunk.document} | Page {chunk.page}")
        
        if show_full_text:
            print(f"    {chunk.text}\n")
        else:
            print(f"    {chunk.text[:150]}...\n")
    
    return results


def analyze_scores(queries: list, top_k: int = 5):
    """
    Analyze similarity scores across multiple queries.
    
    Args:
        queries: List of query strings
        top_k: Number of results per query
    """
    print("Similarity Score Analysis")
    print("=" * 80)
    
    all_scores = []
    for query in queries:
        query_emb = embedder.embed_query(query)
        results = vector_store.search(query_emb, top_k=top_k)
        
        if results:
            scores = [r.score for r in results]
            all_scores.extend(scores)
            print(f"\n{query}")
            print(f"  Top: {max(scores):.3f} | Low: {min(scores):.3f} | Avg: {sum(scores)/len(scores):.3f} | Results: {len(results)}")
    
    if all_scores:
        print("\n" + "=" * 80)
        print(f"Overall: Max={max(all_scores):.3f}, Min={min(all_scores):.3f}, Avg={sum(all_scores)/len(all_scores):.3f}, Std={np.std(all_scores):.3f}")
        print("=" * 80)

### Sample Queries

Test various queries on the corpus.

In [7]:
# Test various queries based on actual document content
query_and_display("¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?", top_k=3)
query_and_display("¿Cuáles son los cuatro ejes del marco estratégico?", top_k=3)
query_and_display("¿Qué organizaciones participaron en el proceso de consulta pública?", top_k=3)
query_and_display("¿Cómo se evalúa la estrategia de accesibilidad universal?", top_k=3)

QUERY: ¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?
Found 3 results (top-3)

[1] Score: 0.8194 | Resumen_grupo_motor_041024.pdf | Page 1
    compromisos del Plan....

[2] Score: 0.7678 | ResumenReunionGA_20231124.pdf | Page 6
    Abierto
II-Proceso de elaboración del Cuarto Plan de Gobierno abierto
Se propone que la elaboración del cuarto plan de Gobierno abierto se desarrolle ...

[3] Score: 0.7595 | Resumen_grupo_motor_14032025.pdf | Page 4
    - Compromiso de evaluación de la estrategia de accesibilidad universal. Por parte de
la directora de la general de accesibilidad se explica que los pr...

QUERY: ¿Cuáles son los cuatro ejes del marco estratégico?
Found 3 results (top-3)

[1] Score: 0.7080 | Resumen_grupo_motor_041024.pdf | Page 1
     En relación con el marco estratégico:
o Se acuerda explicar que el marco estratégico se define precisamente en este
plan de gobierno abierto. El pro...

[2] Score: 0.6422 | Resumen_grupo_motor_14032025.pdf | Page 4
    - Compromi

[SearchResult(chunk=DocumentChunk(text='1. Visión de la accesibilidad universal.\nPara conocer la comprensión del concepto de accesibilidad universal en relación con otros conceptos\nsimilares o de perspectiva más limitada o superada. También las vinculaciones de la accesibilidad con otras\ntemáticas en el ámbito de la ciudad.', document='resumen_grupo_motor110424.pdf', page=9, chunk_index=49), score=0.8077816963195801, rerank_score=None),
 SearchResult(chunk=DocumentChunk(text='La accesibilidad universal es una característica que afecta e implica a todas las unidades organizativas del\nayuntamiento. Para su impulso y desarrollo, el 10 de noviembre de 2022 , se aprueba por acuerdo de Junta\nde Gobierno, el Plan Estratégico de Accesibilidad Universal para la ciudad de Madrid (PEAUM ), como hoja\nde ruta a seguir en los próximos años, que busca una mejora en la gestión con un enfoque transversal y\nuniversal .', document='resumen_grupo_motor110424.pdf', page=8, chunk_index=41), score=0.7

### Score Analysis Across Queries

In [8]:
# Analyze similarity scores across different query types
test_queries = [
    "¿Qué compromisos incluye el cuarto plan?",
    "¿Cuáles son los ejes de transparencia y datos abiertos?",
    "¿Qué es la Escuela de Gobierno Abierto?",
    "¿Quiénes asistieron a las reuniones del grupo motor?",
    "¿Cómo se realiza el seguimiento del plan?",
    "¿Qué papel tiene la participación ciudadana en el plan?",
]

analyze_scores(test_queries, top_k=5)

Similarity Score Analysis

¿Qué compromisos incluye el cuarto plan?
  Top: 0.832 | Low: 0.702 | Avg: 0.744 | Results: 5

¿Cuáles son los ejes de transparencia y datos abiertos?
  Top: 0.837 | Low: 0.613 | Avg: 0.730 | Results: 4

¿Qué es la Escuela de Gobierno Abierto?
  Top: 0.675 | Low: 0.613 | Avg: 0.639 | Results: 3

¿Cómo se realiza el seguimiento del plan?
  Top: 0.708 | Low: 0.676 | Avg: 0.686 | Results: 5

¿Qué papel tiene la participación ciudadana en el plan?
  Top: 0.737 | Low: 0.706 | Avg: 0.717 | Results: 5

Overall: Max=0.837, Min=0.613, Avg=0.708, Std=0.060


---
## 3. Re-ranking with Cross-Encoder

Test the re-ranker to see how it improves results over bi-encoder alone.

### Initialize Re-ranker

In [9]:
from rag_system.reranker import Reranker, RerankerConfig

# Load cross-encoder re-ranker
print("Loading re-ranker...\n")
reranker = Reranker()

print("✓ Re-ranker ready")
print(f"  Model: {reranker.config.model_name}")
print(f"  Top-N: {reranker.config.top_n}")

Loading re-ranker...

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-12-v2...
✓ Cross-encoder loaded
✓ Re-ranker ready
  Model: cross-encoder/ms-marco-MiniLM-L-12-v2
  Top-N: 7


### Compare Bi-encoder vs Cross-encoder Rankings

In [10]:
# Test query with re-ranking
query = "¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?"

# Step 1: Get candidates with bi-encoder (top-20)
query_emb = embedder.embed_query(query)
candidates = vector_store.search(query_emb, top_k=20)

print(f"Query: {query}")
print("=" * 80)
print(f"\nBi-encoder retrieved {len(candidates)} candidates\n")

# Step 2: Re-rank with cross-encoder
reranked = reranker.rerank(query, candidates, top_n=5)

print("Top-5 BEFORE re-ranking (bi-encoder only):")
print("-" * 80)
for i, result in enumerate(candidates[:5], 1):
    print(f"{i}. Score: {result.score:.4f} | {result.chunk.document[:45]}... | Page {result.chunk.page}")

print("\n" + "=" * 80)
print("Top-5 AFTER re-ranking (cross-encoder):")
print("-" * 80)
for i, result in enumerate(reranked, 1):
    print(f"{i}. Bi-encoder: {result.retrieval_score:.4f} | Cross-encoder: {result.rerank_score:.3f}")
    print(f"   {result.chunk.document[:45]}... | Page {result.chunk.page}")

print("\n" + "=" * 80)

Query: ¿Qué compromisos incluye el Cuarto Plan de Gobierno Abierto?

Bi-encoder retrieved 20 candidates

Top-5 BEFORE re-ranking (bi-encoder only):
--------------------------------------------------------------------------------
1. Score: 0.8194 | Resumen_grupo_motor_041024.pdf... | Page 1
2. Score: 0.7678 | ResumenReunionGA_20231124.pdf... | Page 6
3. Score: 0.7595 | Resumen_grupo_motor_14032025.pdf... | Page 4
4. Score: 0.7436 | resumengrupomotor260224.pdf... | Page 1
5. Score: 0.7406 | resumen_grupo_motor110424.pdf... | Page 34

Top-5 AFTER re-ranking (cross-encoder):
--------------------------------------------------------------------------------
1. Bi-encoder: 0.7595 | Cross-encoder: 7.361
   Resumen_grupo_motor_14032025.pdf... | Page 4
2. Bi-encoder: 0.6886 | Cross-encoder: 7.178
   Resumen_grupo_motor_041024.pdf... | Page 1
3. Bi-encoder: 0.7311 | Cross-encoder: 6.849
   resumen_grupo_motor110424.pdf... | Page 3
4. Bi-encoder: 0.6768 | Cross-encoder: 6.350
   resumen_grupo_motor

### Analyze Rank Changes

In [11]:
# Compare rankings and see what changed
comparison = reranker.compare_rankings(query, candidates)

print("Re-ranking Impact Analysis")
print("=" * 80)
print(f"Total candidates:  {comparison['total_candidates']}")
print(f"Top-5 overlap:     {comparison['top_5_overlap']}/5 ({comparison['overlap_percentage']:.0f}%)")

print("\nBiggest rank changes (top-10):")
print("-" * 80)
for change in comparison['rank_changes'][:10]:
    direction = "↑" if change['change'] > 0 else "↓"
    abs_change = abs(change['change'])
    print(f"{direction} Rank {change['old_rank']:2d} → {change['new_rank']:2d} ({abs_change:+2d}): {change['document'][:50]}... | Page {change['page']}")
    print(f"   Bi: {change['retrieval_score']:.3f} | Cross: {change['rerank_score']:.3f}")
    
print("=" * 80)

Re-ranking Impact Analysis
Total candidates:  20
Top-5 overlap:     1/5 (20%)

Biggest rank changes (top-10):
--------------------------------------------------------------------------------
↑ Rank 20 →  5 (+15): resumen_grupo_motor110424.pdf... | Page 1
   Bi: 0.668 | Cross: 6.180
↑ Rank 15 →  2 (+13): Resumen_grupo_motor_041024.pdf... | Page 1
   Bi: 0.689 | Cross: 7.178
↑ Rank 14 →  4 (+10): resumen_grupo_motor110424.pdf... | Page 17
   Bi: 0.677 | Cross: 6.350
↓ Rank  8 → 16 (+8): Resumen_9_reunion_marzo2026.pdf... | Page 1
   Bi: 0.701 | Cross: -0.897
↓ Rank 11 → 19 (+8): Resumen_grupo_motor_041024.pdf... | Page 1
   Bi: 0.819 | Cross: -3.574
↑ Rank 20 → 14 (+6): resumengrupomotor260224.pdf... | Page 2
   Bi: 0.660 | Cross: 2.579
↑ Rank 12 →  7 (+5): resumen_grupo_motor110424.pdf... | Page 24
   Bi: 0.685 | Cross: 5.862
↑ Rank 17 → 12 (+5): resumen_grupo_motor110424.pdf... | Page 2
   Bi: 0.668 | Cross: 4.326
↑ Rank 18 → 13 (+5): resumengrupomotor260224.pdf... | Page 3
   Bi: 0.66

### Inspect Re-ranked Results Content

In [12]:
# Show the actual text of top re-ranked results
print("Top-3 re-ranked results (full text):")
print("=" * 80)

for i, result in enumerate(reranked[:3], 1):
    print(f"\n[{i}] Cross-encoder score: {result.rerank_score:.3f}")
    print(f"    {result.chunk.document} | Page {result.chunk.page}")
    print(f"    {result.chunk.text[:300]}...")
    print("-" * 80)

Top-3 re-ranked results (full text):

[1] Cross-encoder score: 7.361
    Resumen_grupo_motor_14032025.pdf | Page 4
    - Compromiso de evaluación de la estrategia de accesibilidad universal. Por parte de
la directora de la general de accesibilidad se explica que los principales hitos de este
compromiso se iniciarán más adelante de acuerdo con el propio cronograma
incluido en el cuarto plan de gobierno abierto . El p...
--------------------------------------------------------------------------------

[2] Cross-encoder score: 7.178
    Resumen_grupo_motor_041024.pdf | Page 1
    Además, el borrador incluye los antecedentes del tercer plan de gobierno abierto y dos
apartados importantes que se someten también a reflexión del grupo motor. El primero: el
marco estratégico de gobierno abierto que incluye cuatro ejes y una línea transversal.
Los cuatro ejes son: transparencia y ...
--------------------------------------------------------------------------------

[3] Cross-encoder score: 6.849

### Inspect Full Text

In [13]:
# View complete text of retrieved chunks for detailed inspection
query_and_display("¿Qué es el compromiso de comunicación clara?", top_k=2, show_full_text=True)

QUERY: ¿Qué es el compromiso de comunicación clara?
Found 2 results (top-2)

[1] Score: 0.6953 | resumen_grupo-motor240520.pdf | Page 2
    - Importancia de l compromiso de Comunicación clara .
- Aclaración sobre el tipo de sello para certificar que se cumplen los estándares
adecuados en esta materia de comunicación clara .
- Problema de la brecha digital y n ecesidad de mayor accesibilidad en las
comunicaciones con la administración .
- Petición de un mecanismo de comunicación específico para las comunicaciones de
personas mayores con el consejo sectorial de personas mayores. Se trasladó a la
DG mayores dándose una posible solución .

[2] Score: 0.6916 | Resumen_grupo_motor_14032025.pdf | Page 1
    - Compromiso de comunicación clara: se explica por parte de Daniel Vinuesa .
Este compromiso supone un enfoque a corto, medio y largo plazo. En el corto, s e
ha querido comenzar con actuaciones concretas que vayan poco a poco calando en
los profesionales municipales y en la ciudadanía . La

[SearchResult(chunk=DocumentChunk(text='- Importancia de l compromiso de Comunicación clara .\n- Aclaración sobre el tipo de sello para certificar que se cumplen los estándares\nadecuados en esta materia de comunicación clara .\n- Problema de la brecha digital y n ecesidad de mayor accesibilidad en las\ncomunicaciones con la administración .\n- Petición de un mecanismo de comunicación específico para las comunicaciones de\npersonas mayores con el consejo sectorial de personas mayores. Se trasladó a la\nDG mayores dándose una posible solución .', document='resumen_grupo-motor240520.pdf', page=2, chunk_index=7), score=0.6953182816505432, rerank_score=None),
 SearchResult(chunk=DocumentChunk(text='- Compromiso de comunicación clara: se explica por parte de Daniel Vinuesa .\nEste compromiso supone un enfoque a corto, medio y largo plazo. En el corto, s e\nha querido comenzar con actuaciones concretas que vayan poco a poco calando en\nlos profesionales municipales y en la ciudadanía . La co

### Edge Case: Off-Topic Queries

In [14]:
# Test off-topic queries (should return no/low results)
print("Testing Off-Topic Queries (should have low/no results)")
print("=" * 80)

off_topic = [
    "¿Cuál es la receta de la paella valenciana?",
    "¿Qué tiempo hace en Barcelona en verano?",
]

for q in off_topic:
    results = query_and_display(q, top_k=2)
    if results and results[0].score > 0.75:
        print("⚠ WARNING: Unexpectedly high score for off-topic query!\n")
    elif not results:
        print("✓ Correctly filtered: No results above threshold\n")
    else:
        print("✓ Low scores as expected\n")

Testing Off-Topic Queries (should have low/no results)
QUERY: ¿Cuál es la receta de la paella valenciana?
Found 0 results (top-2)

No results found above similarity threshold.

✓ Correctly filtered: No results above threshold

QUERY: ¿Qué tiempo hace en Barcelona en verano?
Found 0 results (top-2)

No results found above similarity threshold.

✓ Correctly filtered: No results above threshold

